In [7]:
from utilities import *

In [8]:
audio = load_audio("/data2/Henri/Journal3/framewiseSpeakerCounting/databases/LibriSpeech/dev-clean/84/121123/84-121123-0000.flac")[0]
print(audio.shape)

torch.Size([1, 33440])


In [9]:
def vad_opt2(
    audio: torch.Tensor, fs: float = 16000.0, thr: float = -30, min_on: float = 50e-3
) -> torch.Tensor:
    # audio is a tensor of shape [num_signals, num_channels, num_samples]

    # Normalisation
    audio = audio - torch.mean(audio, dim=-1, keepdim=True)
    audio = audio / torch.max(torch.abs(audio), dim=-1, keepdim=True)[0]

    # Compute energy in dB
    energy_db = 10 * torch.log10(
        torch.mean(audio.pow(2), dim=1, keepdim=True) + 1e-8
    )  # Add epsilon for stability

    # Detect voice activity using threshold
    vad = torch.ge(energy_db, thr)

    # --- REPLACEMENT START ---
    # Extend each active period efficiently using cumulative sum
    min_period = round(min_on * fs)
    if min_period <= 1:
        return vad

    vad_float = vad.to(audio.dtype)
    num_samples = vad_float.shape[-1]

    # Find the start of each active segment.
    # A segment starts if the current value is 1 and the previous was 0.
    padded_vad = torch.nn.functional.pad(vad_float, (1, 0), "constant", 0)
    is_start = (padded_vad[..., 1:] > padded_vad[..., :-1]).to(audio.dtype)

    # Create markers: +1 at the start of an extension, -1 at the end.
    markers = torch.zeros_like(vad_float)
    markers += is_start  # Add +1 at the start

    # Create the -1 markers for the end of the extension period
    end_markers = torch.nn.functional.pad(is_start, (min_period, 0))[..., :num_samples]
    markers -= end_markers

    # The cumulative sum will create blocks of '1's where VAD should be active.
    vad_extended = torch.cumsum(markers, dim=-1) > 0
    # --- REPLACEMENT END ---

    return vad_extended

In [26]:
def vad_opt2(
    audio: torch.Tensor, fs: float = 16000.0, thr: float = -30, min_on: float = 50e-3
) -> torch.Tensor:
    # audio is a tensor of shape [num_signals, num_channels, num_samples]

    # Normalisation
    audio = audio - torch.mean(audio, dim=-1, keepdim=True)
    audio = audio / torch.max(torch.abs(audio), dim=-1, keepdim=True)[0]

    # Compute energy in dB
    energy_db = 10 * torch.log10(
        torch.mean(audio.pow(2), dim=1, keepdim=True) + 1e-8
    )  # Add epsilon for stability

    # Detect voice activity using threshold
    vad = torch.ge(energy_db, thr)

    # --- REPLACEMENT START ---
    # Extend each active period efficiently using cumulative sum
    min_period = round(min_on * fs)
    if min_period <= 1:
        return vad

    vad_float = vad.to(audio.dtype)
    num_samples = vad_float.shape[-1]

    # Find the start of each active segment.
    # A segment starts if the current value is 1 and the previous was 0.
    padded_vad = torch.nn.functional.pad(vad_float, (1, 0), "constant", 0)
    is_start = (padded_vad[..., 1:] > padded_vad[..., :-1]).to(audio.dtype)

    # Create markers: +1 at the start of an extension, -1 at the end.
    markers = torch.zeros_like(vad_float)
    markers += is_start  # Add +1 at the start

    # Create the -1 markers for the end of the extension period
    end_markers = torch.nn.functional.pad(is_start, (min_period, 0))[..., :num_samples]
    markers -= end_markers

    # The cumulative sum will create blocks of '1's where VAD should be active.
    vad_extended = torch.cumsum(markers, dim=-1) > 0
    # --- REPLACEMENT END ---

    return vad_extended

vad = vad_opt(audio.unsqueeze(0))
vad2 = vad_opt2(audio.unsqueeze(0))
equals = (vad == vad2).sum()
a = torch.where(vad != vad2)
print(a)
print(f"Number of equal elements: {equals} out of {vad.numel()}")
for i in a[2]:
    print(f"Index {i}: vad={vad[0,0,i-2:i+2]}, vad2={vad2[0,0,i-2:i+2]}")

(tensor([0, 0, 0]), tensor([0, 0, 0]), tensor([13573, 29797, 29798]))
Number of equal elements: 33437 out of 33440
Index 13573: vad=tensor([ True,  True,  True, False]), vad2=tensor([ True,  True, False, False])
Index 29797: vad=tensor([True, True, True, True]), vad2=tensor([ True,  True, False, False])
Index 29798: vad=tensor([ True,  True,  True, False]), vad2=tensor([ True, False, False, False])


In [22]:
for i in a[2]:
    print(f"Index {i}: vad={vad[0,0,i-2:i+2]}, vad2={vad2[0,0,i-2:i+2]}")

Index 13573: vad=tensor([ True,  True,  True, False]), vad2=tensor([ True,  True, False, False])
Index 29797: vad=tensor([True, True, True, True]), vad2=tensor([ True,  True, False, False])
Index 29798: vad=tensor([ True,  True,  True, False]), vad2=tensor([ True, False, False, False])
